In [1]:
#!/usr/bin/env python
# coding: utf-8

"""
Position-Wise Accuracy Analysis: Proving Sequential Context Exploitation
=========================================================================

Hypothesis:
-----------
If BiLSTM exploits sequential context, then:
  - Edge positions (near sequence boundaries) should have LOWER accuracy 
    because less bidirectional context is available
  - Center positions should have HIGHER accuracy because full context is available

For position-independent decoders (KL/ML/MinDist), accuracy should be 
UNIFORM across all positions.

This experiment provides direct evidence that the neural decoder's advantage
stems from sequential context exploitation.

Configuration:
--------------
- Alphabet: A_11 (2mix_3mix_4mix, 15 classes)
- Error Model: EZ17 (Erlich)
- Coverage: M = 10
- Sequence Length: 136
"""

# =============================================================================
# IMPORTS
# =============================================================================
import os
import random
import pickle
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import json

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

# =============================================================================
# DEVICE CONFIGURATION
# =============================================================================
DEVICE_ID = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = DEVICE_ID

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

# =============================================================================
# CONFIGURATION
# =============================================================================
CONFIG = {
    # Model Configuration (must match training)
    "error_model": "erlich",
    "error_name": "EZ17",
    "alphabet_mode": "2mix_3mix_4mix",
    "vocab_size": 15,
    "seq_length": 136,
    
    # Coverage to evaluate
    "coverage": 10,
    
    # Model Architecture (must match training)
    "input_channels": 4,
    "hidden_dim": 128,
    "num_layers": 2,
    "dropout": 0.2,
    "bidirectional": True,
    
    # Dataset parameters
    "num_samples": 100000,
    "max_coverage": 25,
    
    # Evaluation
    "batch_size": 500,
    
    # Position regions for analysis
    "edge_size": 15,  # positions 0-14 and (n-15) to (n-1)
    
    # Paths
    "dataset_dir": "./dataset",
    "results_dir": "./results_EZ17_2mix_3mix_4mix",
    "output_dir": "./position_analysis_results",
    
    # Reproducibility
    "seed": 42
}

# Build paths
CONFIG["dataset_path"] = (f"{CONFIG['dataset_dir']}/"
                          f"dna_{CONFIG['error_name']}_{CONFIG['alphabet_mode']}_"
                          f"{CONFIG['num_samples']}_{CONFIG['max_coverage']}.pkl")

CONFIG["model_path"] = (f"{CONFIG['results_dir']}/"
                        f"best_model_{CONFIG['error_name']}_{CONFIG['alphabet_mode']}_"
                        f"M{CONFIG['coverage']}.pth")

os.makedirs(CONFIG['output_dir'], exist_ok=True)

print(f"\n{'='*70}")
print(f"📋 POSITION-WISE ACCURACY ANALYSIS CONFIGURATION")
print(f"{'='*70}")
print(f"   Alphabet: {CONFIG['alphabet_mode']} ({CONFIG['vocab_size']} classes)")
print(f"   Error Model: {CONFIG['error_name']}")
print(f"   Coverage: M = {CONFIG['coverage']}")
print(f"   Sequence Length: {CONFIG['seq_length']}")
print(f"   Edge Region Size: {CONFIG['edge_size']} positions")
print(f"   Dataset Path: {CONFIG['dataset_path']}")
print(f"   Model Path: {CONFIG['model_path']}")
print(f"{'='*70}")

# =============================================================================
# SEED FOR REPRODUCIBILITY
# =============================================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(CONFIG['seed'])

# =============================================================================
# SYMBOL MAPPINGS & IDEAL VECTORS
# =============================================================================
def build_symbol_to_idx():
    """Build symbol-to-index mapping for 2mix_3mix_4mix alphabet."""
    symbol_to_idx = {
        'A': 0, 'C': 1, 'G': 2, 'T': 3,
        'M1': 4, 'M2': 5, 'M3': 6, 'M4': 7, 'M5': 8, 'M6': 9,
        'T1': 10, 'T2': 11, 'T3': 12, 'T4': 13,
        'Q1': 14
    }
    return symbol_to_idx

def build_ideal_vectors():
    """Build ideal frequency vectors for 2mix_3mix_4mix alphabet."""
    third = 1.0 / 3.0
    ideal_vectors = [
        # Pure bases (0-3)
        [1.0, 0.0, 0.0, 0.0],  # A
        [0.0, 1.0, 0.0, 0.0],  # C
        [0.0, 0.0, 1.0, 0.0],  # G
        [0.0, 0.0, 0.0, 1.0],  # T
        # Two-mix (4-9)
        [0.5, 0.0, 0.0, 0.5],  # M1 (A|T)
        [0.0, 0.5, 0.5, 0.0],  # M2 (C|G)
        [0.0, 0.5, 0.0, 0.5],  # M3 (C|T)
        [0.0, 0.0, 0.5, 0.5],  # M4 (G|T)
        [0.5, 0.5, 0.0, 0.0],  # M5 (A|C)
        [0.5, 0.0, 0.5, 0.0],  # M6 (A|G)
        # Three-mix (10-13)
        [third, third, third, 0.0],  # T1 (A|C|G)
        [third, third, 0.0, third],  # T2 (A|C|T)
        [third, 0.0, third, third],  # T3 (A|G|T)
        [0.0, third, third, third],  # T4 (C|G|T)
        # Four-mix (14)
        [0.25, 0.25, 0.25, 0.25],  # Q1 (A|C|G|T)
    ]
    return torch.tensor(ideal_vectors, dtype=torch.float32)

SYMBOL_TO_IDX = build_symbol_to_idx()
IDEAL_VECTORS = build_ideal_vectors().to(device)

# =============================================================================
# DATA PREPROCESSING
# =============================================================================
def preprocess_cluster_to_matrix(cluster_reads, target_length):
    """Convert variable-length noisy reads into frequency matrix."""
    profile_matrix = np.zeros((4, target_length), dtype=np.float32)
    base_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    num_reads = len(cluster_reads)
    
    for read in cluster_reads:
        read_len = len(read)
        if read_len == 0:
            continue
        for t_idx in range(target_length):
            read_idx = int((t_idx + 0.5) * (read_len / target_length))
            if read_idx >= read_len:
                read_idx = read_len - 1
            base = read[read_idx]
            if base in base_map:
                profile_matrix[base_map[base], t_idx] += 1.0
    
    if num_reads > 0:
        profile_matrix /= num_reads
    return profile_matrix

# =============================================================================
# PYTORCH DATASET
# =============================================================================
class CompositeDNADataset(Dataset):
    def __init__(self, data_path, seq_length, symbol_to_idx, limit_coverage=None):
        with open(data_path, 'rb') as f:
            raw_data = pickle.load(f)
        self.samples = raw_data['data']
        self.seq_length = seq_length
        self.symbol_to_idx = symbol_to_idx
        self.limit_coverage = limit_coverage
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        item = self.samples[idx]
        cluster = item['cluster']
        if self.limit_coverage is not None:
            cluster = cluster[:min(self.limit_coverage, len(cluster))]
        x_data = preprocess_cluster_to_matrix(cluster, self.seq_length)
        label_seq = item['label']
        y_data = np.array([self.symbol_to_idx[s] for s in label_seq], dtype=np.longlong)
        return torch.tensor(x_data, dtype=torch.float32), torch.tensor(y_data, dtype=torch.long)

# =============================================================================
# NEURAL NETWORK MODEL (Bi-LSTM)
# =============================================================================
class CompositeDecoderLSTM(nn.Module):
    def __init__(self, config):
        super(CompositeDecoderLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=config['input_channels'],
            hidden_size=config['hidden_dim'],
            num_layers=config['num_layers'],
            batch_first=True,
            bidirectional=config['bidirectional'],
            dropout=config['dropout'] if config['num_layers'] > 1 else 0
        )
        fc_in = config['hidden_dim'] * 2 if config['bidirectional'] else config['hidden_dim']
        self.fc = nn.Linear(fc_in, config['vocab_size'])
        
    def forward(self, x):
        x = x.permute(0, 2, 1)  # (B, 4, L) -> (B, L, 4)
        out, _ = self.lstm(x)
        logits = self.fc(out)
        return logits.permute(0, 2, 1)  # (B, vocab, L)

# =============================================================================
# BASELINE DECODERS
# =============================================================================
def min_distance_decoder(obs, ideal_vectors):
    """Minimum Euclidean Distance Decoder."""
    dists = torch.sum((obs.unsqueeze(2) - ideal_vectors.unsqueeze(0).unsqueeze(0)) ** 2, dim=3)
    return torch.argmin(dists, dim=2)

def kl_divergence_decoder(obs, ideal_vectors, epsilon=0.01):
    """KL Divergence Decoder."""
    ideal_safe = torch.clamp(ideal_vectors.clone(), min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    obs_expanded = obs.unsqueeze(2)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    cross_entropy = -(obs_expanded * log_ideal).sum(dim=-1)
    return torch.argmin(cross_entropy, dim=-1)

def maximum_likelihood_decoder(obs, ideal_vectors, epsilon=0.01):
    """Maximum Likelihood Decoder."""
    ideal_safe = torch.clamp(ideal_vectors.clone(), min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    obs_expanded = obs.unsqueeze(2)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    log_likelihood = (obs_expanded * log_ideal).sum(dim=-1)
    return torch.argmax(log_likelihood, dim=-1)

# =============================================================================
# POSITION-WISE ACCURACY COMPUTATION
# =============================================================================
def compute_position_wise_accuracy(model, loader, ideal_vectors, device, seq_length):
    """
    Compute accuracy at each position for all decoders.
    
    Returns:
        Dictionary with arrays of shape (seq_length,) for each decoder
    """
    model.eval()
    
    # Accumulators: correct counts and total counts per position
    correct_lstm = np.zeros(seq_length)
    correct_mindist = np.zeros(seq_length)
    correct_kl = np.zeros(seq_length)
    correct_ml = np.zeros(seq_length)
    total_per_pos = np.zeros(seq_length)
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            batch_size = inputs.size(0)
            obs = inputs.permute(0, 2, 1)  # (B, L, 4)
            
            # Get predictions from all decoders
            outputs = model(inputs)
            pred_lstm = torch.argmax(outputs, dim=1)  # (B, L)
            pred_mindist = min_distance_decoder(obs, ideal_vectors)
            pred_kl = kl_divergence_decoder(obs, ideal_vectors)
            pred_ml = maximum_likelihood_decoder(obs, ideal_vectors)
            
            # Convert to numpy
            labels_np = labels.cpu().numpy()
            pred_lstm_np = pred_lstm.cpu().numpy()
            pred_mindist_np = pred_mindist.cpu().numpy()
            pred_kl_np = pred_kl.cpu().numpy()
            pred_ml_np = pred_ml.cpu().numpy()
            
            # Accumulate per-position accuracy
            for pos in range(seq_length):
                correct_lstm[pos] += np.sum(pred_lstm_np[:, pos] == labels_np[:, pos])
                correct_mindist[pos] += np.sum(pred_mindist_np[:, pos] == labels_np[:, pos])
                correct_kl[pos] += np.sum(pred_kl_np[:, pos] == labels_np[:, pos])
                correct_ml[pos] += np.sum(pred_ml_np[:, pos] == labels_np[:, pos])
                total_per_pos[pos] += batch_size
    
    # Compute accuracy percentages
    accuracy_lstm = 100 * correct_lstm / total_per_pos
    accuracy_mindist = 100 * correct_mindist / total_per_pos
    accuracy_kl = 100 * correct_kl / total_per_pos
    accuracy_ml = 100 * correct_ml / total_per_pos
    
    return {
        'lstm': accuracy_lstm,
        'mindist': accuracy_mindist,
        'kl': accuracy_kl,
        'ml': accuracy_ml,
        'positions': np.arange(seq_length)
    }

# =============================================================================
# REGION ANALYSIS
# =============================================================================
def analyze_regions(position_accuracies, edge_size, seq_length):
    """
    Analyze accuracy in edge vs center regions.
    
    Edge: positions [0, edge_size) and [seq_length-edge_size, seq_length)
    Center: positions [edge_size, seq_length-edge_size)
    """
    edge_left = list(range(0, edge_size))
    edge_right = list(range(seq_length - edge_size, seq_length))
    edge_positions = edge_left + edge_right
    center_positions = list(range(edge_size, seq_length - edge_size))
    
    results = {}
    for decoder in ['lstm', 'mindist', 'kl', 'ml']:
        acc = position_accuracies[decoder]
        edge_acc = np.mean(acc[edge_positions])
        center_acc = np.mean(acc[center_positions])
        overall_acc = np.mean(acc)
        
        results[decoder] = {
            'edge': edge_acc,
            'center': center_acc,
            'overall': overall_acc,
            'center_minus_edge': center_acc - edge_acc
        }
    
    return results, edge_positions, center_positions

# =============================================================================
# PLOTTING FUNCTIONS
# =============================================================================
def plot_position_wise_accuracy(position_accuracies, save_path, config):
    """Plot position-wise accuracy for all decoders."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    positions = position_accuracies['positions']
    seq_length = config['seq_length']
    edge_size = config['edge_size']
    
    decoders = [
        ('lstm', 'Bi-LSTM (Neural)', '#2ecc71'),
        ('kl', 'KL Divergence', '#3498db'),
        ('ml', 'Max. Likelihood', '#9b59b6'),
        ('mindist', 'Min. Distance', '#e74c3c')
    ]
    
    for ax_idx, (key, name, color) in enumerate(decoders):
        ax = axes[ax_idx // 2, ax_idx % 2]
        acc = position_accuracies[key]
        
        # Plot accuracy curve
        ax.plot(positions, acc, color=color, linewidth=1.5, alpha=0.8)
        
        # Shade edge regions
        ax.axvspan(0, edge_size, alpha=0.2, color='red', label='Edge Region')
        ax.axvspan(seq_length - edge_size, seq_length, alpha=0.2, color='red')
        
        # Add mean lines
        edge_mean = np.mean(np.concatenate([acc[:edge_size], acc[-edge_size:]]))
        center_mean = np.mean(acc[edge_size:-edge_size])
        
        ax.axhline(y=edge_mean, color='red', linestyle='--', linewidth=2, 
                   label=f'Edge Mean: {edge_mean:.2f}%')
        ax.axhline(y=center_mean, color='green', linestyle='--', linewidth=2,
                   label=f'Center Mean: {center_mean:.2f}%')
        
        ax.set_xlabel('Position in Sequence', fontsize=12)
        ax.set_ylabel('Accuracy (%)', fontsize=12)
        ax.set_title(f'{name}', fontsize=14, fontweight='bold')
        ax.legend(fontsize=10, loc='lower right')
        ax.grid(True, alpha=0.3)
        ax.set_xlim(0, seq_length)
        ax.set_ylim(min(acc) - 2, max(acc) + 2)
    
    plt.suptitle(f'Position-Wise Accuracy Analysis\n{config["alphabet_mode"]} ({config["vocab_size"]} classes), '
                 f'{config["error_name"]}, M={config["coverage"]}', fontsize=16)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"📈 Position-wise plot saved: {save_path}")


def plot_comparison_overlay(position_accuracies, save_path, config):
    """Plot all decoders overlaid on single plot."""
    plt.figure(figsize=(14, 6))
    
    positions = position_accuracies['positions']
    seq_length = config['seq_length']
    edge_size = config['edge_size']
    
    # Shade edge regions
    plt.axvspan(0, edge_size, alpha=0.15, color='gray', label='Edge Region')
    plt.axvspan(seq_length - edge_size, seq_length, alpha=0.15, color='gray')
    
    # Plot smoothed curves for better visualization
    from scipy.ndimage import uniform_filter1d
    window = 5
    
    decoders = [
        ('lstm', 'Bi-LSTM (Neural)', '#2ecc71', 2.5),
        ('kl', 'KL / ML', '#3498db', 2.0),
        ('mindist', 'Min. Distance', '#e74c3c', 2.0)
    ]
    
    for key, name, color, lw in decoders:
        acc = position_accuracies[key]
        acc_smooth = uniform_filter1d(acc, size=window)
        plt.plot(positions, acc_smooth, color=color, linewidth=lw, label=name)
    
    plt.xlabel('Position in Sequence', fontsize=14)
    plt.ylabel('Accuracy (%)', fontsize=14)
    plt.title(f'Position-Wise Accuracy: Neural vs Baseline Decoders\n'
              f'{config["alphabet_mode"]} ({config["vocab_size"]} classes), '
              f'{config["error_name"]}, M={config["coverage"]}', fontsize=16)
    plt.legend(fontsize=12, loc='lower center')
    plt.grid(True, alpha=0.3)
    plt.xlim(0, seq_length)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"📈 Comparison overlay plot saved: {save_path}")


def plot_edge_vs_center_bar(region_results, save_path, config):
    """Bar plot comparing edge vs center accuracy."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    decoders = ['lstm', 'mindist', 'kl']
    decoder_names = ['Bi-LSTM', 'Min. Distance', 'KL / ML']
    colors = ['#2ecc71', '#e74c3c', '#3498db']
    
    # Plot 1: Edge vs Center accuracy
    ax1 = axes[0]
    x = np.arange(len(decoders))
    width = 0.35
    
    edge_accs = [region_results[d]['edge'] for d in decoders]
    center_accs = [region_results[d]['center'] for d in decoders]
    
    bars1 = ax1.bar(x - width/2, edge_accs, width, label='Edge Region', color='#e74c3c', alpha=0.8)
    bars2 = ax1.bar(x + width/2, center_accs, width, label='Center Region', color='#2ecc71', alpha=0.8)
    
    ax1.set_ylabel('Accuracy (%)', fontsize=12)
    ax1.set_title('Edge vs Center Region Accuracy', fontsize=14)
    ax1.set_xticks(x)
    ax1.set_xticklabels(decoder_names, fontsize=11)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar in bars1:
        height = bar.get_height()
        ax1.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                     xytext=(0, 3), textcoords="offset points", ha='center', fontsize=10)
    for bar in bars2:
        height = bar.get_height()
        ax1.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                     xytext=(0, 3), textcoords="offset points", ha='center', fontsize=10)
    
    # Plot 2: Center-Edge difference (context exploitation measure)
    ax2 = axes[1]
    diffs = [region_results[d]['center_minus_edge'] for d in decoders]
    
    bars = ax2.bar(x, diffs, width=0.6, color=colors, edgecolor='black', linewidth=1.5)
    
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=1)
    ax2.set_ylabel('Center − Edge Accuracy (%)', fontsize=12)
    ax2.set_title('Context Exploitation Measure\n(Higher = More Context Dependent)', fontsize=14)
    ax2.set_xticks(x)
    ax2.set_xticklabels(decoder_names, fontsize=11)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar, diff in zip(bars, diffs):
        height = bar.get_height()
        va = 'bottom' if height >= 0 else 'top'
        offset = 3 if height >= 0 else -10
        ax2.annotate(f'{diff:+.2f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                     xytext=(0, offset), textcoords="offset points", ha='center', 
                     fontsize=12, fontweight='bold')
    
    plt.suptitle(f'Position-Dependent Accuracy Analysis\n{config["alphabet_mode"]}, '
                 f'{config["error_name"]}, M={config["coverage"]}', fontsize=16)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"📈 Edge vs Center bar plot saved: {save_path}")

# =============================================================================
# PRINT RESULTS
# =============================================================================
def print_results(region_results, config):
    """Print formatted results table."""
    
    print(f"\n{'='*70}")
    print(f"📊 POSITION-WISE ACCURACY ANALYSIS RESULTS")
    print(f"{'='*70}")
    print(f"   Configuration: {config['alphabet_mode']}, {config['error_name']}, M={config['coverage']}")
    print(f"   Sequence Length: {config['seq_length']}")
    print(f"   Edge Region: positions 0-{config['edge_size']-1} and {config['seq_length']-config['edge_size']}-{config['seq_length']-1}")
    print(f"   Center Region: positions {config['edge_size']}-{config['seq_length']-config['edge_size']-1}")
    print(f"{'='*70}")
    
    print(f"\n{'Decoder':<20} {'Edge (%)':<12} {'Center (%)':<12} {'Overall (%)':<12} {'Δ (C-E)':<12}")
    print(f"{'-'*68}")
    
    for decoder in ['lstm', 'kl', 'ml', 'mindist']:
        r = region_results[decoder]
        name = {'lstm': 'Bi-LSTM', 'kl': 'KL Divergence', 'ml': 'Max. Likelihood', 'mindist': 'Min. Distance'}[decoder]
        print(f"{name:<20} {r['edge']:<12.2f} {r['center']:<12.2f} {r['overall']:<12.2f} {r['center_minus_edge']:+.2f}")
    
    print(f"\n{'='*70}")
    print(f"📌 KEY FINDINGS:")
    print(f"{'='*70}")
    
    lstm_diff = region_results['lstm']['center_minus_edge']
    kl_diff = region_results['kl']['center_minus_edge']
    
    print(f"   • Bi-LSTM Center-Edge difference: {lstm_diff:+.2f}%")
    print(f"   • KL/ML Center-Edge difference:   {kl_diff:+.2f}%")
    print(f"   • Difference of differences:      {lstm_diff - kl_diff:+.2f}%")
    
    if lstm_diff > kl_diff + 0.5:
        print(f"\n   ✅ HYPOTHESIS CONFIRMED: Bi-LSTM shows stronger position dependence,")
        print(f"      indicating sequential context exploitation.")
    elif abs(lstm_diff - kl_diff) < 0.5:
        print(f"\n   ⚠️  Results inconclusive: similar position dependence across decoders.")
    else:
        print(f"\n   ❌ Unexpected: KL/ML shows stronger position dependence than Bi-LSTM.")

# =============================================================================
# SAVE RESULTS TO FILE
# =============================================================================
def save_results(position_accuracies, region_results, config, save_path):
    """Save all results to JSON file."""
    results = {
        'config': {k: v for k, v in config.items() if isinstance(v, (int, float, str, list))},
        'position_accuracies': {k: v.tolist() if isinstance(v, np.ndarray) else v 
                                for k, v in position_accuracies.items()},
        'region_results': region_results,
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    
    with open(save_path, 'w') as f:
        json.dump(results, f, indent=4)
    print(f"💾 Results saved: {save_path}")

# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    
    print("\n" + "="*70)
    print("🔬 POSITION-WISE ACCURACY ANALYSIS")
    print("    Proving Sequential Context Exploitation")
    print("="*70)
    
    # Check files exist
    if not os.path.exists(CONFIG['dataset_path']):
        raise FileNotFoundError(f"❌ Dataset not found: {CONFIG['dataset_path']}")
    if not os.path.exists(CONFIG['model_path']):
        raise FileNotFoundError(f"❌ Model not found: {CONFIG['model_path']}")
    
    # Load dataset
    print(f"\n📂 Loading dataset...")
    set_seed(CONFIG['seed'])
    full_ds = CompositeDNADataset(
        CONFIG['dataset_path'],
        CONFIG['seq_length'],
        SYMBOL_TO_IDX,
        limit_coverage=CONFIG['coverage']
    )
    
    # Use test split (20%)
    train_size = int(0.8 * len(full_ds))
    val_size = len(full_ds) - train_size
    _, val_ds = random_split(full_ds, [train_size, val_size])
    val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)
    print(f"   Test samples: {val_size:,}")
    
    # Load model
    print(f"\n🧠 Loading pre-trained Bi-LSTM model...")
    model = CompositeDecoderLSTM(CONFIG).to(device)
    model.load_state_dict(torch.load(CONFIG['model_path'], map_location=device))
    model.eval()
    num_params = sum(p.numel() for p in model.parameters())
    print(f"   Model loaded: {num_params:,} parameters")
    
    # Compute position-wise accuracy
    print(f"\n📊 Computing position-wise accuracy for all decoders...")
    position_accuracies = compute_position_wise_accuracy(
        model, val_loader, IDEAL_VECTORS, device, CONFIG['seq_length']
    )
    
    # Analyze regions
    print(f"\n🔍 Analyzing edge vs center regions...")
    region_results, edge_pos, center_pos = analyze_regions(
        position_accuracies, CONFIG['edge_size'], CONFIG['seq_length']
    )
    
    # Print results
    print_results(region_results, CONFIG)
    
    # Save results
    results_path = os.path.join(CONFIG['output_dir'], 'position_analysis_results.json')
    save_results(position_accuracies, region_results, CONFIG, results_path)
    
    # Generate plots
    print(f"\n📈 Generating plots...")
    
    plot_path1 = os.path.join(CONFIG['output_dir'], 'position_wise_accuracy_4panel.png')
    plot_position_wise_accuracy(position_accuracies, plot_path1, CONFIG)
    
    plot_path2 = os.path.join(CONFIG['output_dir'], 'position_wise_accuracy_overlay.png')
    plot_comparison_overlay(position_accuracies, plot_path2, CONFIG)
    
    plot_path3 = os.path.join(CONFIG['output_dir'], 'edge_vs_center_comparison.png')
    plot_edge_vs_center_bar(region_results, plot_path3, CONFIG)
    
    print(f"\n{'='*70}")
    print(f"✅ ANALYSIS COMPLETE!")
    print(f"{'='*70}")
    print(f"   Output directory: {CONFIG['output_dir']}")
    print(f"   Files generated:")
    print(f"   - position_analysis_results.json")
    print(f"   - position_wise_accuracy_4panel.png")
    print(f"   - position_wise_accuracy_overlay.png")
    print(f"   - edge_vs_center_comparison.png")
    print(f"{'='*70}")

✅ Using device: cuda
   GPU: NVIDIA GeForce RTX 3080

📋 POSITION-WISE ACCURACY ANALYSIS CONFIGURATION
   Alphabet: 2mix_3mix_4mix (15 classes)
   Error Model: EZ17
   Coverage: M = 10
   Sequence Length: 136
   Edge Region Size: 15 positions
   Dataset Path: ./dataset/dna_EZ17_2mix_3mix_4mix_100000_25.pkl
   Model Path: ./results_EZ17_2mix_3mix_4mix/best_model_EZ17_2mix_3mix_4mix_M10.pth

🔬 POSITION-WISE ACCURACY ANALYSIS
    Proving Sequential Context Exploitation

📂 Loading dataset...
   Test samples: 20,000

🧠 Loading pre-trained Bi-LSTM model...
   Model loaded: 536,335 parameters

📊 Computing position-wise accuracy for all decoders...

🔍 Analyzing edge vs center regions...

📊 POSITION-WISE ACCURACY ANALYSIS RESULTS
   Configuration: 2mix_3mix_4mix, EZ17, M=10
   Sequence Length: 136
   Edge Region: positions 0-14 and 121-135
   Center Region: positions 15-120

Decoder              Edge (%)     Center (%)   Overall (%)  Δ (C-E)     
-------------------------------------------------